In [ ]:

import os
import sys
import shutil
import tarfile
import subprocess
import time
from pathlib import Path
from urllib.parse import urlsplit
from urllib.request import Request, urlopen

# ============================================================
# CONFIGURATION
# ============================================================

VERSION = "b12020"
CUDA_VERSION = "12.8"
PORT = 8181

BASE_DIR = Path("/content")
INSTALL_DIR = BASE_DIR / "llama-b12020-bin-cuda"
DOWNLOAD_DIR = BASE_DIR / "llama_downloads"
MODELS_DIR = INSTALL_DIR / "models"

NGROK_AUTH_TOKEN = ""  # Enter your ngrok authtoken here

LLAMA_URL = (
    "https://github.com/ggml-org/llama.cpp/releases/download/"
    "b12020/llama-b12020-bin-ubuntu-cuda-12.8-x64.tar.gz"
)

CUDART_URL = (
    "https://github.com/ggml-org/llama.cpp/releases/download/"
    "b12020/cudart-llama-b12020-bin-ubuntu-cuda-12.8-x64.tar.gz"
)

MODEL_URL = (
    "https://huggingface.co/HauhauCS/"
    "Qwen3.8-27B-Uncensored-HauhauCS-Aggressive-MTP-GGUF/"
    "resolve/main/"
    "Qwen3.8-27B-Uncensored-HauhauCS-Aggressive-Q3_K_P.gguf"
    "?download=true"
)

MMPROJ_URL = (
    "https://huggingface.co/HauhauCS/"
    "Qwen3.8-27B-Uncensored-HauhauCS-Aggressive-MTP-GGUF/"
    "resolve/main/"
    "mmproj-Qwen3.8-27B-Uncensored-HauhauCS-Aggressive-BF16.gguf"
    "?download=true"
)

MODEL_PATH = MODELS_DIR / (
    "Qwen3.8-27B-Uncensored-HauhauCS-Aggressive-Q3_K_P.gguf"
)

MMPROJ_PATH = MODELS_DIR / (
    "mmproj-Qwen3.8-27B-Uncensored-HauhauCS-Aggressive-BF16.gguf"
)

# ============================================================
# UTILITIES
# ============================================================

def run(command, **kwargs):
    print(f"\n$ {' '.join(map(str, command))}", flush=True)
    return subprocess.run(command, check=True, **kwargs)


def download(url, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)

    if destination.exists() and destination.stat().st_size > 0:
        print(f"[SKIP] Already downloaded: {destination.name}")
        return

    print(f"[DOWNLOAD] {destination.name}", flush=True)

    temporary = destination.with_name(destination.name + ".part")

    try:
        import requests
        from tqdm.auto import tqdm

        headers = {}
        existing = temporary.stat().st_size if temporary.exists() else 0

        if existing:
            headers["Range"] = f"bytes={existing}-"

        with requests.get(
            url, stream=True, timeout=(30, 120), headers=headers
        ) as response:
            response.raise_for_status()

            append = existing > 0 and response.status_code == 206
            if not append:
                existing = 0

            total = int(response.headers.get("Content-Length", 0))

            with open(temporary, "ab" if append else "wb") as f:
                with tqdm(
                    total=total + existing if total else None,
                    initial=existing,
                    unit="B",
                    unit_scale=True,
                    desc=destination.name,
                ) as bar:
                    for chunk in response.iter_content(1024 * 1024):
                        if chunk:
                            f.write(chunk)
                            bar.update(len(chunk))

        temporary.replace(destination)

    except Exception:
        print(f"[ERROR] Download failed: {destination.name}")
        raise


def extract_archive(archive, destination):
    print(f"[EXTRACT] {archive.name}")
    destination.mkdir(parents=True, exist_ok=True)

    with tarfile.open(archive, "r:gz") as tar:
        # Prevent archive members from escaping the destination.
        tar.extractall(destination, filter="data")


def merge_contents(source, destination):
    """Recursively merge source files into destination."""
    destination.mkdir(parents=True, exist_ok=True)

    for item in source.iterdir():
        target = destination / item.name

        if item.is_dir() and target.is_dir():
            merge_contents(item, target)
            item.rmdir()
        else:
            if target.is_file() or target.is_symlink():
                target.unlink()
            elif target.is_dir():
                shutil.rmtree(target)

            shutil.move(str(item), str(target))


def find_executable(directory, name):
    matches = list(directory.rglob(name))
    matches = [p for p in matches if p.is_file()]

    if not matches:
        raise FileNotFoundError(f"{name} not found in {directory}")

    return matches[0]


# ============================================================
# STEP 1: CHECK NVIDIA GPU
# ============================================================

print("\n========== GPU INFORMATION ==========")

run(["nvidia-smi"])

# ============================================================
# STEP 2: INSTALL PYTHON DEPENDENCIES
# ============================================================

print("\n========== INSTALL DEPENDENCIES ==========")

run([
    sys.executable, "-m", "pip", "install", "-q",
    "requests", "tqdm", "pyngrok"
])

from pyngrok import ngrok

# ============================================================
# STEP 3: CREATE DIRECTORIES
# ============================================================

INSTALL_DIR.mkdir(parents=True, exist_ok=True)
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# STEP 4: DOWNLOAD LLAMA.CPP AND CUDA RUNTIME
# ============================================================

print("\n========== DOWNLOAD LLAMA.CPP ==========")

llama_archive = DOWNLOAD_DIR / "llama-cuda.tar.gz"
cudart_archive = DOWNLOAD_DIR / "cudart-cuda.tar.gz"

download(LLAMA_URL, llama_archive)
download(CUDART_URL, cudart_archive)

# ============================================================
# STEP 5: EXTRACT ARCHIVES
# ============================================================

print("\n========== EXTRACT ARCHIVES ==========")

llama_extract = DOWNLOAD_DIR / "llama_extracted"
cudart_extract = DOWNLOAD_DIR / "cudart_extracted"

extract_archive(llama_archive, llama_extract)
extract_archive(cudart_archive, cudart_extract)

# ============================================================
# STEP 6: MERGE CUDA RUNTIME INTO LLAMA.CPP
# ============================================================

print("\n========== MERGE CUDA LIBRARIES ==========")

merge_contents(llama_extract, INSTALL_DIR)
merge_contents(cudart_extract, INSTALL_DIR)

print("[OK] CUDA runtime files merged.")

# ============================================================
# STEP 7: DOWNLOAD GGUF MODEL AND VISION PROJECTOR
# ============================================================

print("\n========== DOWNLOAD MODELS ==========")

download(MODEL_URL, MODEL_PATH)
download(MMPROJ_URL, MMPROJ_PATH)

print("[OK] Model files downloaded.")

# ============================================================
# STEP 8: LOCATE LLAMA SERVER
# ============================================================

print("\n========== FIND LLAMA SERVER ==========")

SERVER_PATH = find_executable(INSTALL_DIR, "llama-server")
SERVER_PATH.chmod(SERVER_PATH.stat().st_mode | 0o111)

print(f"Server executable: {SERVER_PATH}")

# Make shared CUDA libraries discoverable by Linux.
library_dirs = {
    str(SERVER_PATH.parent),
    str(INSTALL_DIR),
}

for lib in INSTALL_DIR.rglob("*.so*"):
    library_dirs.add(str(lib.parent))

env = os.environ.copy()
env["LD_LIBRARY_PATH"] = (
    ":".join(sorted(library_dirs))
    + ":" + env.get("LD_LIBRARY_PATH", "")
)

# ============================================================
# STEP 9: START LLAMA.CPP SERVER
# ============================================================

print("\n========== START LLAMA SERVER ==========")

server_command = [
    str(SERVER_PATH),
    "-m", str(MODEL_PATH),
    "--mmproj", str(MMPROJ_PATH),
    "-ngl", "999",
    "--host", "127.0.0.1",
    "--port", str(PORT),
]

LOG_PATH = INSTALL_DIR / "llama-server.log"

log_file = open(LOG_PATH, "w")

server_process = subprocess.Popen(
    server_command,
    cwd=str(SERVER_PATH.parent),
    env=env,
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

print(f"[OK] Server process started. PID: {server_process.pid}")

# ============================================================
# STEP 10: WAIT FOR SERVER READINESS
# ============================================================

print("\n========== WAIT FOR SERVER ==========")

import requests

ready = False

for attempt in range(120):
    if server_process.poll() is not None:
        log_file.flush()
        print(LOG_PATH.read_text(errors="replace")[-6000:])
        raise RuntimeError("llama-server exited unexpectedly.")

    try:
        response = requests.get(
            f"http://127.0.0.1:{PORT}/health",
            timeout=3
        )

        if response.status_code == 200:
            ready = True
            break

    except requests.RequestException:
        pass

    if attempt % 10 == 0:
        print(f"Waiting for model loading... {attempt * 5}s")

    time.sleep(5)

if not ready:
    raise TimeoutError(
        f"Server did not become ready. Check {LOG_PATH}"
    )

print("[OK] llama.cpp server is ready.")

# ============================================================
# STEP 11: CREATE NGROK TUNNEL
# ============================================================

print("\n========== START NGROK ==========")

if not NGROK_AUTH_TOKEN.strip():
    from getpass import getpass
    NGROK_AUTH_TOKEN = getpass("Enter ngrok authtoken: ")

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Close existing tunnels from this notebook runtime.
ngrok.kill()

tunnel = ngrok.connect(
    addr=PORT,
    proto="http",
)

PUBLIC_URL = tunnel.public_url

# ============================================================
# STEP 12: DISPLAY RESULTS
# ============================================================

print("\n" + "=" * 60)
print("LLAMA.CPP DEPLOYMENT COMPLETE")
print("=" * 60)

print(f"Installation: {INSTALL_DIR}")
print(f"Model:        {MODEL_PATH}")
print(f"Vision:       {MMPROJ_PATH}")
print(f"Local API:    http://127.0.0.1:{PORT}")
print(f"Public API:   {PUBLIC_URL}")
print(f"API endpoint: {PUBLIC_URL}/v1/chat/completions")
print(f"Logs:         {LOG_PATH}")

print("=" * 60)
print("Keep this Colab runtime running to maintain the server.")
